# Classificador de Risco para Triagem Clínica

Neste notebook, vamos construir um classificador utilizando NLP (TF-IDF) e Machine Learning para sugerir o grau de risco ("baixo risco" ou "alto risco") de forma automatizada com base no relato do paciente.

In [2]:
!pip install pandas scikit-learn

  Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.7 MB 16.7 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 22.9 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 23.2 MB/s  0:00:00
Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------- ----------------------- 5.0/12.3 MB 26.0 MB/s eta 0:00:01
   ------------------------------------- -- 11.5/12.3 MB 28.3 MB/s eta 0:00:01
   ---

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

### 1. Carregamento dos Dados
Vamos carregar a base de dados (`base_risco.csv`) contendo frases rotuladas com o risco do paciente.

In [4]:
# Carregando os dados
df = pd.read_csv('base_risco.csv')
display(df.head())

,frase,situacao
0,sinto dor no peito e falta de ar,alto risco
1,tive um leve incômodo nas costas,baixo risco
2,dor aguda no braço esquerdo acompanhada de sud...,alto risco
3,estou com uma tosse seca há alguns dias,baixo risco
4,desmaiei e meu coração estava batendo muito rá...,alto risco


### 2. Transformação de Texto com TF-IDF
O TF-IDF (Term Frequency - Inverse Document Frequency) transforma o texto das frases em formato vetorial.

In [5]:
# Inicializando o TfidfVectorizer
vetorizador = TfidfVectorizer()

# Aplicar a vetorização nos dados de texto
X = vetorizador.fit_transform(df['frase'])
y = df['situacao']

print(f"Tamanho da matriz vetorial: {X.shape}")

Tamanho da matriz vetorial: (20, 96)


### 3. Divisão de Treino e Teste
Separando os dados em base de treinamento e validação para testar a acurácia do algoritmo em dados não vistos.

In [6]:
# Dividindo 70% para treino e 30% para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### 4. Treinamento do Modelo
Vamos utilizar a Regressão Logística, que funciona muito bem para classificação binária simples como o nosso caso de uso.

In [7]:
# Treinando do Modelo Logistic Regression
modelo = LogisticRegression()
modelo.fit(X_train, y_train)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


### 5. Avaliação do Modelo
Vamos prever os valores da base de teste para validar nossa acurácia observando possíveis vieses em frases muito específicas.

In [8]:
# Predições usando o conjunto de teste
y_pred = modelo.predict(X_test)

# Acurácia Geral
acuracia = accuracy_score(y_test, y_pred)
print(f"Acurácia do Modelo: {acuracia * 100:.2f}%\n")

# Relatório Completo
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred))

Acurácia do Modelo: 33.33%

Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       0.33      1.00      0.50         2
 baixo risco       0.00      0.00      0.00         4

    accuracy                           0.33         6
   macro avg       0.17      0.50      0.25         6
weighted avg       0.11      0.33      0.17         6



### 6. Teste Prático
Vamos simular o funcionamento recebendo dados novos não mapeados no nosso arquivo CSV para ver se a predição acompanha o padrão sintomático.

In [9]:
# Experimente colocar uma frase nova
novas_frases = [
    "acordei com uma pressão forte e estranha no peito e sinto meu coração acelerado",
    "estou com os olhos um pouco vermelhos e dor na perna",
    "sinto muita falta de ar e fraqueza"
]

vetores_novos = vetorizador.transform(novas_frases)
previsoes = modelo.predict(vetores_novos)

print("==================== RESULTADO NA TRIAGEM ====================")
for frase, pred in zip(novas_frases, previsoes):
    print(f"Relato: '{frase}' \n>>> IA Direciona para: {pred.upper()}\n")

==================== RESULTADO NA TRIAGEM ====================
Relato: 'acordei com uma pressão forte e estranha no peito e sinto meu coração acelerado' 
>>> IA Direciona para: ALTO RISCO

Relato: 'estou com os olhos um pouco vermelhos e dor na perna' 
>>> IA Direciona para: ALTO RISCO

Relato: 'sinto muita falta de ar e fraqueza' 
>>> IA Direciona para: ALTO RISCO

